# ☁️ MiniStack Control Center (EC2 & S3 Dashboard)
Aplikasi interaktif berbasis **Jupyter Notebook (ipywidgets)** untuk memanajemen resource **AWS EC2** dan **S3** pada emulator local (Ministack / LocalStack).

---
### 🛠️ Persiapan Library
Jalankan sel di bawah ini untuk menginstal dependensi yang dibutuhkan jika belum terinstall.

In [1]:
%pip install boto3 ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ 1. Inisialisasi Client & Logger
Konfigurasi koneksi Boto3 ke endpoint LocalStack/Ministack (`http://localhost:4566`) serta pembuatan komponen UI untuk menampilkan log aktivitas.

In [2]:
import boto3
from botocore.exceptions import ClientError
import ipywidgets as widgets
from IPython.display import display, clear_output

# Konfigurasi Endpoint Emulator
ENDPOINT = 'http://localhost:4566'
REGION = 'us-east-1'
CREDENTIALS = {
    'aws_access_key_id': '121212121212',
    'aws_secret_access_key': 'admin123'
}

# Inisialisasi Boto3 Clients
ec2_client = boto3.client('ec2', endpoint_url=ENDPOINT, region_name=REGION, **CREDENTIALS)
s3_client = boto3.client('s3', endpoint_url=ENDPOINT, region_name=REGION, **CREDENTIALS)

# Komponen Log System UI
out_log = widgets.Output()

def show_log(msg, success=True):
    with out_log:
        clear_output()
        color = "green" if success else "red"
        display(widgets.HTML(f"<b style='color:{color};'>[{ 'INFO' if success else 'ERROR' }] {msg}</b>"))

print("Client Boto3 & System Log siap digunakan.")

Client Boto3 & System Log siap digunakan.


## ⚡ 2. Komponen Modul EC2 Instance
Bagian ini mengatur form untuk meluncurkan *Instance* EC2 baru serta menampilkan daftar instance aktif.

In [3]:
# Input Widgets untuk EC2
ec2_ami_input = widgets.Text(value='ami-mock-ubuntu', description='AMI ID:')
ec2_type_input = widgets.Dropdown(options=['t2.micro', 't3.medium', 'c8g.medium'], value='t2.micro', description='Type:')
btn_launch_ec2 = widgets.Button(description='Launch Instance', button_style='success', icon='rocket')

ec2_table_out = widgets.Output()

def refresh_ec2_list():
    with ec2_table_out:
        clear_output()
        try:
            res = ec2_client.describe_instances()
            instances = []
            for resv in res.get('Reservations', []):
                for inst in resv.get('Instances', []):
                    instances.append(f"🆔 ID: {inst['InstanceId']} | 🟢 State: {inst['State']['Name']} | 💻 Type: {inst['InstanceType']}")
            
            if instances:
                print("\n".join(instances))
            else:
                print("Tidak ada EC2 Instance yang aktif.")
        except Exception as e:
            print(f"Gagal mengambil data EC2: {e}")

def on_launch_ec2_clicked(b):
    try:
        res = ec2_client.run_instances(
            ImageId=ec2_ami_input.value,
            InstanceType=ec2_type_input.value,
            MinCount=1,
            MaxCount=1
        )
        inst_id = res['Instances'][0]['InstanceId']
        show_log(f"Berhasil membuat EC2 Instance: {inst_id}")
        refresh_ec2_list()
    except Exception as e:
        show_log(f"Gagal meluncurkan EC2: {e}", success=False)

btn_launch_ec2.on_click(on_launch_ec2_clicked)

ec2_box = widgets.VBox([
    widgets.HTML("<h3>⚡ EC2 Resource Provisioning</h3>"),
    widgets.HBox([ec2_ami_input, ec2_type_input, btn_launch_ec2]),
    widgets.HTML("<hr><b>Daftar EC2 Instances:</b>"),
    ec2_table_out
])

print("Modul EC2 berhasil dimuat.")

Modul EC2 berhasil dimuat.


## 🪣 3. Komponen Modul S3 Bucket & Storage
Bagian ini menyediakan antarmuka interaktif untuk membuat Bucket, memilih Bucket aktif, dan mengunggah teks langsung sebagai objek S3.

In [4]:
# Input Widgets untuk S3
s3_bucket_input = widgets.Text(value='my-app-bucket', description='Bucket Name:')
btn_create_bucket = widgets.Button(description='Create Bucket', button_style='primary', icon='folder-plus')

s3_bucket_select = widgets.Dropdown(description='Select Bucket:')
btn_refresh_buckets = widgets.Button(description='Refresh Buckets', icon='sync')

file_key_input = widgets.Text(value='notes.txt', description='File Name:')
file_content_input = widgets.Textarea(value='Halo dari Dashboard Ministack!', description='Content:')
btn_upload_file = widgets.Button(description='Upload File', button_style='warning', icon='upload')

s3_files_out = widgets.Output()

def refresh_bucket_dropdown():
    try:
        res = s3_client.list_buckets()
        buckets = [b['Name'] for b in res.get('Buckets', [])]
        s3_bucket_select.options = buckets
    except Exception as e:
        show_log(f"Gagal memuat daftar bucket: {e}", success=False)

def refresh_s3_files():
    with s3_files_out:
        clear_output()
        target_bucket = s3_bucket_select.value
        if not target_bucket:
            print("Pilih bucket terlebih dahulu.")
            return
        
        try:
            res = s3_client.list_objects_v2(Bucket=target_bucket)
            if 'Contents' in res:
                print(f"--- Isi dari Bucket '{target_bucket}' ---")
                for obj in res['Contents']:
                    print(f"📄 {obj['Key']} ({obj['Size']} bytes)")
            else:
                print(f"Bucket '{target_bucket}' masih kosong.")
        except Exception as e:
            print(f"Gagal membaca isi bucket: {e}")

def on_create_bucket_clicked(b):
    try:
        s3_client.create_bucket(Bucket=s3_bucket_input.value)
        show_log(f"Bucket '{s3_bucket_input.value}' berhasil dibuat!")
        refresh_bucket_dropdown()
    except Exception as e:
        show_log(f"Gagal membuat bucket: {e}", success=False)

def on_upload_file_clicked(b):
    target_bucket = s3_bucket_select.value
    if not target_bucket:
        show_log("Pilih bucket tujuan terlebih dahulu!", success=False)
        return
    try:
        s3_client.put_object(
            Bucket=target_bucket,
            Key=file_key_input.value,
            Body=file_content_input.value.encode('utf-8')
        )
        show_log(f"File '{file_key_input.value}' berhasil diunggah ke '{target_bucket}'!")
        refresh_s3_files()
    except Exception as e:
        show_log(f"Gagal mengunggah file: {e}", success=False)

btn_create_bucket.on_click(on_create_bucket_clicked)
btn_refresh_buckets.on_click(lambda b: refresh_bucket_dropdown())
btn_upload_file.on_click(on_upload_file_clicked)
s3_bucket_select.observe(lambda change: refresh_s3_files() if change['name'] == 'value' else None)

s3_box = widgets.VBox([
    widgets.HTML("<h3>🪣 S3 Object Storage Dashboard</h3>"),
    widgets.HBox([s3_bucket_input, btn_create_bucket]),
    widgets.HTML("<hr>"),
    widgets.HBox([s3_bucket_select, btn_refresh_buckets]),
    widgets.VBox([
        file_key_input,
        file_content_input,
        btn_upload_file
    ]),
    widgets.HTML("<hr><b>Daftar File dalam Bucket:</b>"),
    s3_files_out
])

print("Modul S3 berhasil dimuat.")

Modul S3 berhasil dimuat.


## 🚀 4. Peluncuran Antarmuka (UI Dashboard)
Menggabungkan semua modul ke dalam tampilan berbasis Tab Interaktif. Jalankan sel ini untuk menampilkan Dashboard.

In [5]:
# Gabungkan modul EC2 dan S3 ke dalam Navigasi Tab
tab = widgets.Tab()
tab.children = [ec2_box, s3_box]
tab.set_title(0, 'EC2 Manager')
tab.set_title(1, 'S3 Manager')

# Tampilkan UI utama
display(widgets.HTML("<h2>☁️ MiniStack Control Center (EC2 & S3)</h2>"))
display(tab)
display(out_log)

# Refresh tampilan data awal
refresh_ec2_list()
refresh_bucket_dropdown()

HTML(value='<h2>☁️ MiniStack Control Center (EC2 & S3)</h2>')

Output()